In [ ]:
from ucimlrepo import fetch_ucirepo 
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import optuna
import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Device: {device}")
print(f"CUDA Version: {torch.version.cuda}")

In [ ]:
adult = fetch_ucirepo(id=2)
df = pd.concat([adult.data.features, adult.data.targets], axis=1)
df = df.fillna('undefined')

df['income'] = df['income'].str.replace('.', '', regex=False).str.strip()
df['capital-difference'] = df['capital-gain'] - df['capital-loss']
df['age_group'] = pd.cut(df['age'], bins=[16, 25, 35, 45, 65], labels=['young', 'adult', 'middle_age', 'senior'])
df.drop(['fnlwgt'], axis=1, inplace=True)

for col in df.select_dtypes(exclude=['int64', 'float64']).columns:
    df[col] = LabelEncoder().fit_transform(df[col])

X = df.drop(['income'], axis=1)
y = (df['income'] > 0).astype(int)

X = RobustScaler().fit_transform(X)

x_train, x_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
x_val, x_test, y_val, y_test = train_test_split(x_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

def to_tensor(x, y):
    return torch.tensor(x, dtype=torch.float32), torch.tensor(y.values, dtype=torch.float32)

x_train, y_train = to_tensor(x_train, y_train)
x_val, y_val = to_tensor(x_val, y_val)
x_test, y_test = to_tensor(x_test, y_test)

train_dataset = TensorDataset(x_train, y_train)
val_dataset = TensorDataset(x_val, y_val)
test_dataset = TensorDataset(x_test, y_test)

In [ ]:
class CensusModel(nn.Module):
    def __init__(self, input_dim, hidden_layers, hidden_size, dropout_rate):
        super().__init__()
        layers = []
        for i in range(hidden_layers):
            layers.append(nn.Linear(input_dim if i == 0 else hidden_size, hidden_size))
            layers.append(nn.BatchNorm1d(hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
        layers.append(nn.Linear(hidden_size, 1))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)


In [ ]:
def objective(trial):
    hidden_layers = trial.suggest_int("hidden_layers", 2, 6)
    hidden_size = trial.suggest_int("hidden_size", 128, 512)
    dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.6)
    lr = trial.suggest_float("lr", 1e-5, 5e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [64, 128, 256])
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "RMSprop"])
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)

    model = CensusModel(x_train.shape[1], hidden_layers, hidden_size, dropout_rate).to(device)
    optimizer = getattr(optim, optimizer_name)(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)

    # Handle imbalance
    class_counts = np.bincount(y_train.int().cpu().numpy())
    pos_weight = torch.tensor([class_counts[0] / class_counts[1]]).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    best_val_acc = 0
    patience = 5
    trigger = 0

    for epoch in range(50):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb).squeeze(), yb)
            loss.backward()
            optimizer.step()

        model.eval()
        val_preds, val_labels = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                preds = (torch.sigmoid(model(xb).squeeze()) > 0.5).float()
                val_preds.extend(preds.cpu().numpy())
                val_labels.extend(yb.cpu().numpy())

        acc = accuracy_score(val_labels, val_preds)
        scheduler.step(acc)

        if acc > best_val_acc:
            best_val_acc = acc
            trigger = 0
        else:
            trigger += 1
            if trigger >= patience:
                break  # early stop

    return best_val_acc


In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=40, show_progress_bar=True)

print("\n=== Best Hyperparameters ===")
print(study.best_trial.params)
print(f"Best Validation Accuracy: {study.best_value:.4f}")


In [ ]:
best_params = study.best_trial.params
final_model = CensusModel(
    x_train.shape[1],
    best_params["hidden_layers"],
    best_params["hidden_size"],
    best_params["dropout_rate"]
).to(device)

optimizer = getattr(optim, best_params["optimizer"])(final_model.parameters(), lr=best_params["lr"])
criterion = nn.BCEWithLogitsLoss()
train_loader = DataLoader(train_dataset, batch_size=best_params["batch_size"], shuffle=True)

for epoch in range(40):
    final_model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(final_model(xb).squeeze(), yb)
        loss.backward()
        optimizer.step()


In [ ]:
final_model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for xb, yb in DataLoader(test_dataset, batch_size=128):
        xb, yb = xb.to(device), yb.to(device)
        preds = (torch.sigmoid(final_model(xb).squeeze()) > 0.5).float()
        y_true.extend(yb.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

print("\n=== Final Classification Report ===")
print(classification_report(y_true, y_pred, digits=4))
print("=== Confusion Matrix ===")
print(confusion_matrix(y_true, y_pred))
